**PHÂN CỤM DỮ LIỆU VÀ PHÁT HIỆN ẢNH TRÙNG**

In [1]:
import numpy as np
import pandas as pd
import os

from PIL import Image, UnidentifiedImageError
from tensorflow.keras.applications import MobileNet
from tensorflow.keras.applications.mobilenet import preprocess_input
from tensorflow.keras.preprocessing import image as keras_image
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
file_path = "D:\\VSCode\\CS114\\CarDataset.csv" # Đường dẫn file dữ liệu csv
dataset_path = "D:\\Dataset\\FinalData_CS114"   # Đường dẫn thư mục cha chứa các thư mục ảnh hãng xe
dataset = pd.read_csv(file_path)

model = MobileNet(weights='imagenet', include_top=False, pooling='avg', input_shape=(224, 224, 3))

In [3]:
def batch_extract_features(image_paths):
    batch_images = []   # Danh sách lưu mảng np của các ảnh hợp lệ
    valid_paths = []    # Danh sách lưu đường dẫn của các ảnh hợp lệ 

    for img_path in image_paths:    # Xử lý từng ảnh 
        try:
            img = keras_image.load_img(img_path, target_size=(224, 224))
            if img.mode != 'RGB':
                img = img.convert('RGBA').convert('RGB')
            img_array = keras_image.img_to_array(img)
            batch_images.append(img_array)
            valid_paths.append(img_path)
        except (FileNotFoundError, UnidentifiedImageError):
            print(f"Skipping invalid or missing image: {img_path}")
        except Exception as e:
            print(f"Error processing {img_path}: {e}")

    # Xử lý batch ảnh hợp lệ
    if batch_images:
        batch_images = np.array(batch_images)
        batch_images = preprocess_input(batch_images)   # Tiền xử lý ảnh để phù hợp với MobileNet (chuẩn hóa pixel về [-1, 1])
        features = model.predict(batch_images)  # Dự đoán để trích xuất đặc trưng từ mô hình MobileNet
        return features, valid_paths    # Trả về đặc trưng (features) và danh sách các ảnh hợp lệ
    else:
        return np.array([]), []

In [4]:
# Khởi tạo batch size và danh sách xử lý
batch_size = 32 # Xử lý ảnh theo từng nhóm 32 ảnh 
features = []   # Danh sách lưu đặc trưng ảnh sau khi trích xuất
processed_image_paths = []  # Danh sách lưu đường dẫn của ảnh sau khi xử lý
image_paths = dataset['ImageFullPath'].apply(lambda x: os.path.join(dataset_path, x)).tolist()
for i in range(0, len(image_paths), batch_size):
    batch_paths = image_paths[i:i + batch_size]
    batch_features, valid_paths = batch_extract_features(batch_paths)
    features.append(batch_features)
    processed_image_paths.extend(valid_paths)

# Gộp đặc trưng và tạo Dataframe
features = np.vstack(features) if features else np.array([])    # Gộp danh sách đặc trưng tất cả các batch thành một mảng numpy duy nhất.

# Tạo DataFrame chứa đường dẫn ảnh hợp lệ và cột ClusterID (chưa được gán giá trị, ban đầu là None)
filtered_dataset = pd.DataFrame({
    'ImageFullPath': processed_image_paths,
    'ClusterID': None
})

if features.size > 0:
    pca = PCA(n_components=500)
    features_reduced = pca.fit_transform(features)

    kmeans = KMeans(n_clusters=9, random_state=42)
    kmeans.fit(features_reduced)
    labels = kmeans.labels_

    filtered_dataset['ClusterID'] = labels

output_file = "clustering_results.csv"
filtered_dataset.to_csv(output_file, index=False)
print(f"Clustering results saved to {output_file}")

# Nhóm các ảnh theo ClusterID trong filtered_dataset
# Ví dụ: 
"""
post_check = {
    0: (np.array([[0.1, 0.2], [0.3, 0.4]]), ['img1.jpg', 'img2.jpg']),
    1: (np.array([[0.5, 0.6], [0.7, 0.8]]), ['img3.jpg', 'img4.jpg'])
}
"""
post_check = filtered_dataset.groupby('ClusterID').apply(
    lambda group: (features[group.index], group['ImageFullPath'].tolist())
)

dupe_results = []

def find_duplicates_within_cluster(cluster_features, image_paths, similarity_threshold=0.95):
    duplicates = []
    # Tìm các cặp ảnh trùng lặp trong một cụm dựa trên độ tương đồng cosine:
    similarity_matrix = cosine_similarity(cluster_features)

    # So sánh từng cặp ảnh trong cụm
    # Nếu độ tương đồng > similarity_threshold, thêm vào danh sách trùng lặp
    for i in range(len(cluster_features)):
        for j in range(i + 1, len(cluster_features)):
            if similarity_matrix[i, j] > similarity_threshold:
                duplicates.append({
                    'Image1': image_paths[i],
                    'Image2': image_paths[j],
                    'Similarity': similarity_matrix[i, j]
                })
    return duplicates

#  Ghi kết quả trùng lặp
# Tập refined_potential_duplicates là danh sách các cặp ảnh cụ thể được xác định là trùng lặp tiềm năng dựa trên ngưỡng độ tương đồng cosine
for cluster_id, (cluster_features, image_list) in post_check.items():
    if len(image_list) > 1:
        cluster_duplicates = find_duplicates_within_cluster(cluster_features, image_list)
        for dupe in cluster_duplicates:
            dupe_results.append({'ClusterID': cluster_id, **dupe})

dupe_results_df = pd.DataFrame(dupe_results)
dupe_output_file = "refined_potential_duplicates.csv"
dupe_results_df.to_csv(dupe_output_file, index=False)
print(f"Refined potential duplicates saved to {dupe_output_file}")

dupe_results_df = pd.DataFrame(dupe_results)

# Tạo tập hậu kiểm và lọc dữ liệu
# Tập Hậu Kiểm là danh sách các ảnh được xác định là trùng lặp tiềm năng dựa trên ngưỡng độ tương đồng cosine
all_duplicate_paths = set(dupe_results_df['Image1']).union(set(dupe_results_df['Image2']))

hau_kiem_dataset = filtered_dataset[filtered_dataset['ImageFullPath'].isin(all_duplicate_paths)]
hau_kiem_dataset.to_csv("Hậu Kiểm.csv", index=False)
print("Duplicates saved to Hậu Kiểm.csv")

filtered_dataset = filtered_dataset[~filtered_dataset['ImageFullPath'].isin(all_duplicate_paths)]
filtered_dataset.to_csv("clustering_results.csv", index=False)
print("Filtered dataset saved to clustering_results.csv")

1/1 [==============================] - 0s 265ms/step
Skipping invalid or missing image: D:\Dataset\FinalData_CS114\Others\22520477-22520490.Others.8.jpg
1/1 [==============================] - 0s 291ms/step


c:\Users\LENOVO\AppData\Local\Programs\Python\Python311\Lib\site-packages\PIL\Image.py:1054: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


1/1 [==============================] - 0s 408ms/step
Skipping invalid or missing image: D:\Dataset\FinalData_CS114\Honda\22521560-22521614.Honda.41.jpg
1/1 [==============================] - 0s 338ms/step
Skipping invalid or missing image: D:\Dataset\FinalData_CS114\Mitsubishi\22521463-22521213-22521259.Mitsubishi.76.jpg
1/1 [==============================] - 0s 310ms/step
Skipping invalid or missing image: D:\Dataset\FinalData_CS114\Mitsubishi\22521463-22521213-22521259.Mitsubishi.57.jpg
1/1 [==============================] - 0s 368ms/step
Skipping invalid or missing image: D:\Dataset\FinalData_CS114\Others\22521463-22521213-22521259.Others.53.jpg
1/1 [==============================] - 0s 353ms/step
Skipping invalid or missing image: D:\Dataset\FinalData_CS114\Others\22520348-22520530-22520837.MG.15.jpg
1/1 [==============================] - 0s 294ms/step
Skipping invalid or missing image: D:\Dataset\FinalData_CS114\Others\22520477-22520490.Others.7.jpg
1/1 [==========================

: 

In [25]:
tag_mapping = {
    "Honda": 1,
    "Hyundai": 2,
    "KIA": 3,
    "Mazda": 4,
    "Mitsubishi": 5,
    "Suzuki": 6,
    "Toyota": 7,
    "VinFast": 8,
    "Others": 0
}

In [26]:
clusterd_data = "D:\\VSCode\\CS114\\clustering_results.csv"
df = pd.read_csv(clusterd_data)

In [27]:
def extract_brand_name(file_path):
    parts = file_path.split('\\')
    for part in parts:
        if part in tag_mapping:
            return part
    return "Others"

In [28]:
df['BrandName'] = df['ImageFullPath'].apply(extract_brand_name)
df['Label'] = df['BrandName'].map(tag_mapping)

In [29]:
output_file_path = 'labeled_clustering_results.csv'
df.to_csv(output_file_path, index=False)

**Xóa hai cột ClusterID và BrandName của file labeled_clustering_results.csv**

In [2]:
import pandas as pd

# Đường dẫn tới file CSV gốc
input_csv_file = "D:\\VSCode\\CS114\\labeled_clustering_results.csv"

# Đường dẫn tới file CSV sau khi xử lý
output_csv_file = "D:\\VSCode\\CS114\\final_114cardataset.csv"

# Đọc file CSV vào DataFrame
df = pd.read_csv(input_csv_file)

# Xóa các cột 'ClusterID' và 'BrandName'
columns_to_drop = ['ClusterID', 'BrandName']
df = df.drop(columns=columns_to_drop, errors='ignore')

# Ghi DataFrame đã xử lý vào file CSV mới
df.to_csv(output_csv_file, index=False)

print(f"File CSV đã được xử lý và lưu tại: {output_csv_file}")


File CSV đã được xử lý và lưu tại: D:\VSCode\CS114\final_114cardataset.csv
